In [2]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np

# ── 1. Data ──────────────────────────────────────────────────────────────────
X, y = make_classification(
    n_samples=1000, n_features=4, n_redundant=2,
    n_classes=2, random_state=42
)

# ── 2. Preprocessing  ────────────────────────────────────────────────────────
# KNN is distance-based → scaling is essential
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── 3. Train / Test Split ────────────────────────────────────────────────────
# Fixed: test_size should be a float (proportion), not an int of 30
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ── 4. Hyperparameter Tuning via GridSearchCV ────────────────────────────────
param_grid = {
    'n_neighbors': list(range(1, 21)),          # k from 1 → 20
    'weights'    : ['uniform', 'distance'],
    'metric'     : ['euclidean', 'manhattan', 'minkowski'],
    'p'          : [1, 2]                        # power for Minkowski
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,                  # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,             # use all CPU cores
    verbose=1
)
grid_search.fit(X_train, y_train)

print("── Best Parameters ──────────────────────────────")
print(grid_search.best_params_)
print(f"Best CV Accuracy : {grid_search.best_score_:.4f}")

# ── 5. Evaluate Best Model ────────────────────────────────────────────────────
best_model = grid_search.best_estimator_
y_pred     = best_model.predict(X_test)

print("\n── Test-Set Metrics ─────────────────────────────")
print(f"Accuracy         : {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix :")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report :")
print(classification_report(y_test, y_pred))

# ── 6. Cross-Validation Score on Best Model ───────────────────────────────────
cv_scores = cross_val_score(best_model, X_scaled, y, cv=5, scoring='accuracy')
print(f"\nCross-Val Accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Fitting 5 folds for each of 240 candidates, totalling 1200 fits
── Best Parameters ──────────────────────────────
{'metric': 'euclidean', 'n_neighbors': 7, 'p': 1, 'weights': 'distance'}
Best CV Accuracy : 0.9050

── Test-Set Metrics ─────────────────────────────
Accuracy         : 0.9300

Confusion Matrix :
[[97  4]
 [10 89]]

Classification Report :
              precision    recall  f1-score   support

           0       0.91      0.96      0.93       101
           1       0.96      0.90      0.93        99

    accuracy                           0.93       200
   macro avg       0.93      0.93      0.93       200
weighted avg       0.93      0.93      0.93       200


Cross-Val Accuracy : 0.9100 ± 0.0300
